# Day 4 — Winter ONI forecast only

Feed the latest 12 Pacific SST anomaly maps into the trained CNN → save **`forecast.json`**.

**Impacts are not precomputed here.** Day 5 map clicks call a live API (`cloud/impacts_api.py`) that pulls Open-Meteo for that lat/lon and fits ONI→winter on the fly.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aarib-sami/ninonet/blob/main/enso/day4_impacts.ipynb)

Need a Day 3 checkpoint (`enso_cnn_lead6_tuned.pt` preferred).


## 0. Install


In [ ]:
!pip install -q xarray netCDF4 numpy pandas torch


## 1. Mount Drive


In [ ]:
from pathlib import Path

from google.colab import drive
drive.mount("/content/drive")

DATA_DIR = Path("/content/drive/MyDrive/ensocast/data")
OUT_DIR = Path("/content/drive/MyDrive/ensocast/artifacts")
OUT_DIR.mkdir(parents=True, exist_ok=True)

assert (DATA_DIR / "pacific_anom.nc").exists(), "Rerun Day 1"
assert (DATA_DIR / "oni_monthly.csv").exists(), "Rerun Day 1"
print("Data:", DATA_DIR)
print("Artifacts:", OUT_DIR)


## 2. Load anomalies


In [ ]:
import numpy as np
import pandas as pd
import xarray as xr

anom = xr.open_dataarray(DATA_DIR / "pacific_anom.nc")
if isinstance(anom, xr.Dataset):
    anom = anom[list(anom.data_vars)[0]]

arr = anom.values.astype("float32")
times = pd.to_datetime(anom["time"].values).to_period("M").to_timestamp()

oni_df = pd.read_csv(DATA_DIR / "oni_monthly.csv", parse_dates=["time"])
oni_series = oni_df.set_index("time")["oni"]
oni_series.index = pd.to_datetime(oni_series.index).to_period("M").to_timestamp()
oni = oni_series.reindex(times).to_numpy(dtype="float32")

missing = int(np.isnan(oni).sum())
if missing:
    valid = ~np.isnan(oni)
    arr, times, oni = arr[valid], times[valid], oni[valid]
    print(f"Dropped {missing} month(s) with no ONI")

print("months:", len(arr), "last:", times[-1].date(), "last ONI:", float(oni[-1]))


## 3. Load CNN + forecast


In [ ]:
import torch
import torch.nn as nn

LEAD = 6
WINDOW = 12
WINTER_LABEL = "2026-27"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


class ENSOForecaster(nn.Module):
    def __init__(self, in_months=12, dropout=0.4, use_bn=True):
        super().__init__()
        layers = [nn.Conv2d(in_months, 32, 3, padding=1)]
        if use_bn:
            layers.append(nn.BatchNorm2d(32))
        layers += [nn.ReLU(), nn.MaxPool2d(2), nn.Conv2d(32, 64, 3, padding=1)]
        if use_bn:
            layers.append(nn.BatchNorm2d(64))
        layers += [
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 1),
        ]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(-1)


candidates = [
    OUT_DIR / f"enso_cnn_lead{LEAD}_tuned.pt",
    OUT_DIR / f"enso_cnn_lead{LEAD}.pt",
    OUT_DIR / "enso_cnn_lead3_tuned.pt",
    OUT_DIR / "enso_cnn_lead3.pt",
]
ckpt_path = next((p for p in candidates if p.exists()), None)
assert ckpt_path is not None, "No checkpoint — rerun Day 2/3"
print("Loading", ckpt_path)

ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
state = ckpt["model_state"]
use_bn = any("running_mean" in k for k in state)
dropout = float(ckpt.get("hp", {}).get("dropout", 0.4)) if isinstance(ckpt.get("hp"), dict) else 0.4
model = ENSOForecaster(dropout=dropout, use_bn=use_bn).to(device)
model.load_state_dict(state)
model.eval()

x = arr[-WINDOW:].astype("float32")
mu, sd = ckpt.get("norm_mu"), ckpt.get("norm_sd")
if mu is not None and sd is not None:
    x = (x - float(mu)) / float(sd)

with torch.no_grad():
    forecast_oni = float(model(torch.from_numpy(x[None]).to(device)).cpu().numpy().squeeze())

last_month = times[-1]
print(f"input ends: {last_month.date()}")
print(f"forecast ONI (winter {WINTER_LABEL}): {forecast_oni:.3f}")


## 4. Write `forecast.json`

Day 5 API reads this + `oni_monthly.csv`. Copy both somewhere the API can see (Drive download, or `data/` / `artifacts/` locally).


In [ ]:
import json
import shutil

payload = {
    "winter": WINTER_LABEL,
    "forecast_oni": float(forecast_oni),
    "model_checkpoint": ckpt_path.name,
    "model_lead": int(ckpt.get("lead", LEAD)),
    "input_end": str(last_month.date()),
    "last_observed_oni": float(oni[-1]),
    "disclaimer": (
        "CNN ONI forecast only. Per-location winter impacts are computed live on map click "
        "via Open-Meteo + historical ONI regression — not an official outlook."
    ),
}

for path in [OUT_DIR / "forecast.json", DATA_DIR / "forecast.json"]:
    path.write_text(json.dumps(payload, indent=2), encoding="utf-8")
    print("Wrote", path)

# Keep a Drive copy of ONI next to forecast for the API
print("ONI CSV still at", DATA_DIR / "oni_monthly.csv")
print("Day 4 checkpoint: forecast.json ready (no precomputed city grid).")
print(json.dumps(payload, indent=2))
